# Notebook 01 — Baseline Predictive Logistic Regression Model

**Project:** Predictive Power Management in IoT Devices Using Low-Complexity Machine Learning and Dynamic Scheduling  
**Student:** Rishabh Yadav | A00048105  
**Supervisor:** Jennifer McManis  


---

## Purpose

This notebook implements the **baseline predictive logistic regression model** for next step occupancy prediction.  
The model learns to predict whether a space will be **occupied at time t+1** given environmental sensor readings **at time t**.

This is fundamentally different from simply classifying current occupancy.  
The **t+1 formulation** is what enables the IoT device to make **proactive** sleep scheduling decisions,  
rather than reacting to the current state after the fact.

## Dataset

UCI Occupancy Detection Dataset — Candanedo & Feldheim (2016)  
Source: https://archive.ics.uci.edu/dataset/357/occupancy+detection  
Sensors: Temperature, Humidity, Light, CO2, HumidityRatio  
Sampling interval: 1 minute  
Ground truth: Binary occupancy label (0 = unoccupied, 1 = occupied)

## Prediction Task

```
X(t) = [Temperature, Humidity, Light, CO2, HumidityRatio,
         hour, minute, Light_lag1, CO2_lag1,
         Light_delta, CO2_delta]  (11 features)

y = Occupancy(t+1)  -- next-step binary label
```

Target shifting: `df['Occupancy_t_plus_1'] = df['Occupancy'].shift(-1)`

## Model

Logistic Regression — chosen for TinyML feasibility on ARM Cortex-M class devices  
Justified by: Warden & Situnayake (2019) TinyML constraints (<50KB, <5ms inference)

## Outputs

- Trained logistic regression model
- Accuracy, AUC-ROC, classification report
- Confusion matrix
- Feature importance (model coefficients)
- Saved predictions CSV for downstream energy simulation

---
## Section 1 — Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score
)

print('Libraries loaded successfully.')

---
## Section 2 — Load Dataset

In [ ]:
train_path = '../data/datatraining.txt'
test_path  = '../data/datatest2.txt'   # larger test set used for final evaluation

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

# Convert date column
train_df['date'] = pd.to_datetime(train_df['date'])
test_df['date']  = pd.to_datetime(test_df['date'])

print('Train shape:', train_df.shape)
print('Test shape :', test_df.shape)
print('Columns    :', train_df.columns.tolist())
print('\nClass distribution (train):')
print(train_df['Occupancy'].value_counts())
print(f'Occupancy rate: {train_df["Occupancy"].mean():.1%}')

---
## Section 3 — Feature Engineering

Sixteen features are engineered from five raw sensor readings.

**Why these features?**
- **Lag features** — capture what conditions were one minute ago (recent history)
- **Delta features** — capture whether conditions are rising or falling (rate of change)
- **Time features** — hour and minute capture daily occupancy patterns (Candanedo & Feldheim, 2016)
- **Work hour flag** — binary indicator for standard working hours (08:00–18:00)

**Target shifting** — `Occupancy.shift(-1)` maps current readings to next-step label,  
enabling the model to predict what will happen rather than what is happening.

In [ ]:
def add_features(df):
    """
    Engineer features for next-step occupancy prediction.
    Returns dataframe with 11 input features and Occupancy_t1 target.
    """
    df = df.copy()

    # --- Time features ---
    df['hour']         = df['date'].dt.hour
    df['minute']       = df['date'].dt.minute
    df['is_work_hour'] = ((df['hour'] >= 8) & (df['hour'] <= 18)).astype(int)

    # --- Lag features (previous time step) ---
    df['Light_lag1'] = df['Light'].shift(1)
    df['CO2_lag1']   = df['CO2'].shift(1)
    df['Temp_lag1']  = df['Temperature'].shift(1)

    # --- Delta features (rate of change) ---
    df['Light_delta'] = df['Light'] - df['Light_lag1']
    df['CO2_delta']   = df['CO2']   - df['CO2_lag1']
    df['Temp_delta']  = df['Temperature'] - df['Temp_lag1']

    # --- Rolling mean (3-step window for noise reduction) ---
    df['Light_roll3'] = df['Light'].rolling(3).mean()
    df['CO2_roll3']   = df['CO2'].rolling(3).mean()

    # --- NEXT-STEP TARGET: X(t) -> Occupancy(t+1) ---
    # This shift is the core of the predictive formulation.
    # The model learns to predict the NEXT minute's occupancy,
    # not the current minute's. This enables proactive scheduling.
    df['Occupancy_t1'] = df['Occupancy'].shift(-1)

    # Drop rows with NaN from lag/shift operations
    df = df.dropna().reset_index(drop=True)
    df['Occupancy_t1'] = df['Occupancy_t1'].astype(int)

    return df


train_df = add_features(train_df)
test_df  = add_features(test_df)

print('Processed train shape:', train_df.shape)
print('Processed test shape :', test_df.shape)

# --- Define feature set and target ---
FEATURE_COLS = [
    'Temperature', 'Humidity', 'Light', 'CO2', 'HumidityRatio',
    'hour', 'minute', 'is_work_hour',
    'Light_lag1', 'CO2_lag1', 'Temp_lag1',
    'Light_delta', 'CO2_delta', 'Temp_delta',
    'Light_roll3', 'CO2_roll3'
]
TARGET = 'Occupancy_t1'

print(f'\nNumber of features: {len(FEATURE_COLS)}')
print(f'Features: {FEATURE_COLS}')

X_train = train_df[FEATURE_COLS]
y_train = train_df[TARGET]
X_test  = test_df[FEATURE_COLS]
y_test  = test_df[TARGET]

print(f'\nTraining samples : {len(X_train)}')
print(f'Test samples     : {len(X_test)}')
print(f'Occupied (train) : {y_train.sum()} ({y_train.mean():.1%})')
print(f'Occupied (test)  : {y_test.sum()} ({y_test.mean():.1%})')

---
## Section 4 — Feature Scaling and Model Training

StandardScaler is applied before Logistic Regression because the features have  
very different scales (Light 0–1000 lux vs Temperature 18–25°C).  
Scaling ensures no single feature dominates the weight vector due to scale alone.

In [ ]:
# --- Scale features ---
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# --- Train Logistic Regression ---
# Logistic Regression selected because:
# - Model size ~0.86 KB (well within 50 KB TinyML limit)
# - Inference time ~146 µs (well within 5 ms limit)
# - Interpretable coefficients for feature importance
# Reference: Warden & Situnayake (2019) TinyML
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_sc, y_train)

print('Model trained successfully.')
print(f'Model parameters: {model.coef_.shape[1]} weights + 1 bias term')

---
## Section 5 — Prediction and Evaluation

The model outputs:
- `y_pred` — binary class label (0 or 1) for Occupancy(t+1)
- `y_prob` — probability P(Occupancy(t+1) = 1)

**P is the key output for the scheduler.**  
The adaptive sleep scheduler uses P — not the binary label — to make  
fine-grained decisions between Active, Light Sleep, and Deep Sleep states.

In [ ]:
# --- Generate predictions ---
y_pred = model.predict(X_test_sc)
y_prob = model.predict_proba(X_test_sc)[:, 1]  # P(Occupancy at t+1 = 1)

# --- Evaluation metrics ---
acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print('=' * 50)
print('NEXT-STEP PREDICTION RESULTS  X(t) -> Occupancy(t+1)')
print('=' * 50)
print(f'Accuracy : {acc:.4f}  ({acc:.1%})')
print(f'AUC-ROC  : {auc:.4f}')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred,
      target_names=['Unoccupied (0)', 'Occupied (1)']))

---
## Section 6 — Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm,
    display_labels=['Unoccupied (0)', 'Occupied (1)']).plot(ax=ax, colorbar=False)
ax.set_title(
    'Confusion Matrix — Next-Step Occupancy Prediction\n'
    'Logistic Regression  |  X(t) -> Occupancy(t+1)',
    fontsize=11
)
plt.tight_layout()
plt.savefig('../figures/confusion_matrix_baseline.png', dpi=150)
plt.show()
print('Saved: ../figures/confusion_matrix_baseline.png')

---
## Section 7 — Feature Importance

The magnitude of the logistic regression coefficients indicates  
how strongly each feature influences the prediction.  
This directly validates the finding of Candanedo & Feldheim (2016)  
that Light and CO2 are the most discriminative predictors of occupancy.

In [ ]:
coef_df = pd.DataFrame({
    'Feature'    : FEATURE_COLS,
    'Coefficient': model.coef_[0]
})
coef_df['Abs_Coefficient'] = coef_df['Coefficient'].abs()
coef_df = coef_df.sort_values('Abs_Coefficient', ascending=False).reset_index(drop=True)

print('Feature Importance (Logistic Regression Coefficients):')
print(coef_df[['Feature', 'Coefficient']].to_string(index=False))

# --- Plot feature importance ---
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#D62728' if c > 0 else '#1F77B4'
          for c in coef_df['Coefficient']]
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Logistic Regression Coefficient')
ax.set_title(
    'Feature Importance — Next-Step Occupancy Prediction\n'
    'Red = positive influence on occupancy, Blue = negative'
)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/feature_importance_baseline.png', dpi=150)
plt.show()
print('Saved: ../figures/feature_importance_baseline.png')

---
## Section 8 — Exploratory Visualisation

Visualising the relationship between sensor readings and occupancy  
confirms the patterns the model has learned.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

train_df.groupby('Occupancy')['Light'].mean().plot(
    kind='bar', ax=axes[0], color=['#1F77B4', '#D62728'], edgecolor='black'
)
axes[0].set_title('Average Light by Occupancy Status')
axes[0].set_xlabel('Occupancy (0 = Unoccupied, 1 = Occupied)')
axes[0].set_ylabel('Average Light (lux)')
axes[0].tick_params(axis='x', rotation=0)

train_df.groupby('Occupancy')['CO2'].mean().plot(
    kind='bar', ax=axes[1], color=['#1F77B4', '#D62728'], edgecolor='black'
)
axes[1].set_title('Average CO2 by Occupancy Status')
axes[1].set_xlabel('Occupancy (0 = Unoccupied, 1 = Occupied)')
axes[1].set_ylabel('Average CO2 (ppm)')
axes[1].tick_params(axis='x', rotation=0)

plt.suptitle(
    'Sensor-Occupancy Relationship Validation\n'
    '(Confirms Candanedo & Feldheim, 2016 findings)',
    fontsize=11
)
plt.tight_layout()
plt.savefig('../figures/sensor_occupancy_relationship.png', dpi=150)
plt.show()
print('Saved: ../figures/sensor_occupancy_relationship.png')

---
## Section 9 — Save Predictions for Downstream Notebooks

The prediction results (predicted labels and probabilities) are saved  
as a CSV file for use in the energy model simulation (Notebook 03)  
and the threshold sensitivity analysis (Notebook 04).

**Note:** The probability column `y_prob` is the key output —  
the adaptive scheduler uses P to make threshold-based state decisions,  
not the binary label directly.

In [ ]:
results_df = test_df.copy()
results_df['y_pred'] = y_pred
results_df['y_prob'] = y_prob  # P(Occupancy at t+1 = 1) -- used by scheduler

results_df.to_csv('../data/best_model_predictions.csv', index=False)

print('Saved: ../data/best_model_predictions.csv')
print(f'Total rows saved : {len(results_df)}')
print(f'y_prob range     : {y_prob.min():.4f} to {y_prob.max():.4f}')
print(f'Median y_prob    : {np.median(y_prob):.4f}')
print()
print('This file is the input to:')
print('  - Notebook 03: Energy model and power state simulation')
print('  - Notebook 04: Threshold sensitivity analysis')

---
## Summary

| Metric | Value |
|--------|-------|
| Prediction task | X(t) → Occupancy(t+1) |
| Number of features | 16 |
| Model | Logistic Regression |
| Test accuracy | ~97.8% |
| AUC-ROC | ~0.995 |
| Training samples | 8,141 |
| Test samples | 9,749 |

**Key finding:** Logistic Regression achieves near-identical AUC to more complex models  
(Random Forest AUC = 0.997) while being 39× smaller and 11× faster —  
making it the optimal choice for TinyML IoT deployment.

**Next notebook:** `02_model_comparison.ipynb` — formal comparison of all four  
candidate models with TinyML feasibility analysis.